In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sklearn.metrics import mean_squared_error

In [4]:
# Step 1: Load the data
ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

# View the structure of the dataset
print(ratings.head())
print(movies.head())

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
   adult                              belongs_to_collection    budget  \
0  False  {'id': 10194, 'name': 'Toy Story Collection', ...  30000000   
1  False                                                NaN  65000000   
2  False  {'id': 119050, 'name': 'Grumpy Old Men Collect...         0   
3  False                                                NaN  16000000   
4  False  {'id': 96871, 'name': 'Father of the Bride Col...         0   

                                              genres  \
0  [{'id': 16, 'name': 'Animation'}, {'id': 35, '...   
1  [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...   
2  [{'id': 10749, 'name': 'Romance'}, {'id': 35, ...   
3  [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...   
4                     [{'id': 35, 'name': 'Comedy'}] 

C:\Users\muhid\AppData\Local\Temp\ipykernel_14000\2471968290.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv('movies.csv')


In [5]:
# Step 2: Create user item matrix
# We’ll create a matrix where rows represent users, columns represent movies,
# and the values are the ratings given by the users to the movies.
# Missing values indicate that the user hasn’t rated the movie yet.
user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating')

# Fill the missing value with 0.
user_item_matrix.fillna(0, inplace=True)
#print(user_item_matrix

In [6]:
# Step 3: Implement Collaborative filtering.
# We’ll use Cosine Similarity to find similar users or items.
# For user-based collaborative filtering, we calculate the similarity between users,
# and for item-based filtering, we calculate the similarity between movies.
# Let’s use item-based collaborative filtering (we recommend movies similar to those the user already liked):
# Example: Recommend a comedy movie because user has already liked a comedy movie.
# We will use cosine similarity becuase it focusses on pattern similarity rather than absolute ratings.

# Calculate cosine similarity between movies.
item_simlarity = cosine_similarity(user_item_matrix.T)
# This matrix shows which movies were rated by which users, and the ratings given.
# We did transpose because:When calculating item-based similarity, we want to compare movies (items) with each other, not users.
item_simlarity_df = pd.DataFrame(item_simlarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)


In [7]:
# Step 4: Make predictions based on similarity
# We are using item_similarity calculated above to predict the ratings of the movies the users have not rated yet.
# We do so by using cosine_similarity, that is:
# If two movies are similar, the rating for one can help predict the rating for the other.
# For example, if a user rated Movie A and Movie B is very similar to Movie A, we can predict how the user might rate Movie B based on their rating of Movie A.
def predict_ratings(user_item_matrix, item_similarity_matrix):
    return np.dot(user_item_matrix, item_similarity_matrix) / np.abs(item_similarity_matrix).sum(axis=1)

# Make prediction
predicted_ratings = predict_ratings(user_item_matrix.values, item_simlarity) # which pass the user ratings for each movie
# Convert it to Dataframe for better readability:
predicted_ratings_df = pd.DataFrame(predicted_ratings, index=user_item_matrix.index, columns=user_item_matrix.columns)
# print(predicted_ratings_df)

In [8]:

# Step 4: Evaluate the model
# Flatten the matrix and calculate root mean square error.
true_ratings = user_item_matrix.values.flatten()
predicted_ratings = predicted_ratings_df.values.flatten()

# Calculate root mean square error:
rmse = mean_squared_error(true_ratings[true_ratings > 0], predicted_ratings[true_ratings > 0])
print("The root mean square error: ", rmse)

The root mean square error:  9.895247046806295
